In [1]:
import numpy as np
import anndata
from scipy import sparse

In [2]:
basepath = 'data/tasic/'
adata_tasic = anndata.read_h5ad(f'{basepath}adata.h5ad')

In [4]:
adata_tasic.shape

(23822, 42776)

In [3]:
def simulate(adata,only_marker_genes_DE,read_umi_factor = 100, theta_umi = 100, theta_z = 1, seed = 42):

    cluster_ids = np.unique(adata.obs['clusters'])

    readcounts_sim = []
    umi_counts_sim = []
    clusterlabels_sim = []
    clustercolors_sim = []

    housekeeping_idx = ~adata.var['marker_idx']
    across_cluster_ps = np.array(adata.X.sum(axis=0) / adata.X.sum()).flatten()

    for cluster_i in cluster_ids:

        print(f'simulating cluster {cluster_i}..')

        cluster_idx = adata.obs['clusters'] == cluster_i
        ad_cl = adata[cluster_idx]

        ###get UMIs
        #get seqdepths per cell
        readcount_ns = np.array(ad_cl.X.sum(axis=1)).flatten()
        umi_ns = readcount_ns / read_umi_factor
        #get proportions per gene
        ps = np.array(ad_cl.X.sum(axis=0) / np.sum(readcount_ns)).flatten()
        
        if only_marker_genes_DE:
            #set ps to across-cluster ps for all non-marker genes
            ps[housekeeping_idx] = across_cluster_ps[housekeeping_idx]

        #sample UMIs from NB(n*p)
        mus = np.outer(umi_ns,ps)
        nb_p = theta_umi / (theta_umi + mus)
        np.random.seed(seed)
        umi_counts = np.random.negative_binomial(np.ones_like(nb_p)*theta_umi, nb_p)

        ###amplify UMIs to get readcounts
        #intialize readcounts
        readcounts=np.zeros(umi_counts.shape)
        #only nonzero counts need amplification observations 
        umi_nonzero_idx = umi_counts>0
        umi_nonzero_counts = umi_counts[umi_nonzero_idx]
        #index for which amplification sample(s) get to amplify which UMIs
        split_idx = np.cumsum(umi_nonzero_counts).astype(int)
        #one large sample instead of separate ones is more efficient (one sample per molecule observed)
        
        mean_z = read_umi_factor
        np.random.seed(seed)        
        zs = np.random.geometric(1/mean_z,size=sum(umi_nonzero_counts))

        #splitting up into separate groups of samples for each (gene-x-cell)-observation
        zs_per_cell_x_gene=np.split(zs,split_idx[:-1])
        #summing reads for each (gene-x-cell)-observation
        zs_sums =[sum(z) for z in zs_per_cell_x_gene]
        #mapping back to count matrix
        readcounts[umi_nonzero_idx]=zs_sums

        n_cells = sum(cluster_idx)
        clusterlabels_sim += [cluster_i]*n_cells
        clustercolors_sim += [adata.uns['clustercolors'][cluster_i]]*n_cells
        readcounts_sim += [readcounts]
        umi_counts_sim += [umi_counts]
    
    adata_sim = anndata.AnnData(X=sparse.csc_matrix(np.concatenate(readcounts_sim)))
    adata_sim.obs['clustercolor'] = clustercolors_sim
    adata_sim.obs['clusters'] = clusterlabels_sim
    adata_sim.layers['umis_sim'] = sparse.csc_matrix(np.concatenate(umi_counts_sim))
    adata_sim.uns['theta_umi']=theta_umi
    adata_sim.uns['theta_z']=theta_z
    adata_sim.uns['seed']=seed
    adata_sim.uns['read_umi_factor']=read_umi_factor
    adata_sim.var_names = adata.var_names.copy()
    adata_sim.var['marker_idx'] = adata.var['marker_idx'].copy()
    
    return adata_sim


In [4]:
theta_umi = 100
theta_z = 1

In [5]:
adata_sim = simulate(adata_tasic,theta_umi=theta_umi,theta_z=theta_z,only_marker_genes_DE=False)
adata_sim_only_marker_genes_DE = simulate(adata_tasic,theta_umi=theta_umi,theta_z=theta_z,only_marker_genes_DE=True)

simulating cluster 0..
simulating cluster 1..
simulating cluster 2..
simulating cluster 3..



KeyboardInterrupt



In [6]:
#add length info to adatas
adata_sim.var['transcript_len_max'] = adata_tasic.var['transcript_len_max'].copy()
adata_sim_only_marker_genes_DE.var['transcript_len_max'] = adata_tasic.var['transcript_len_max'].copy()

adata_sim.write_h5ad(f'data/tasic/simulations/adata_sim_thetaUMI_{theta_umi}_GeomZ.h5ad')
adata_sim_only_marker_genes_DE.write_h5ad(f'data/tasic/simulations/adata_sim_only_marker_genes_DE_thetaUMI_{theta_umi}_GeomZ.h5ad')

#### Save TPM-normalized counts for further processing with census/qUMI

In [7]:
from scipy.io import mmwrite
import pandas as pd
basepath = 'data/tasic/simulations/'

def save_tpm_mtx_and_tpm_adata(adata,simname='simulation'):

    #get genes for which we have length info
    adata_tpm = adata.copy()[:,~np.isnan(adata.var['transcript_len_max'])]
    print(f'{simname}: subsetting adata to genes with known length, new shape: {adata_tpm.shape}')

    #convert counts to TPM
    reads_per_kb = adata_tpm.X / (adata_tpm.var['transcript_len_max'].values/1000)
    adata_tpm.layers['tpm'] = (reads_per_kb / reads_per_kb.sum(axis=1)) * 1e6
    
    tpm_df = adata_tpm.to_df(layer='tpm')
    #save transposed sparse version for further processing in R
    mmwrite(f'{basepath}{simname}_tpm_transposed.mtx',sparse.csr_array(tpm_df.T))
    pd.DataFrame(tpm_df.columns).to_csv(f'{basepath}{simname}_tpm_genes.csv')
    pd.DataFrame(tpm_df.index).to_csv(f'{basepath}{simname}_tpm_cells.csv')

    #also write out adata
    adata_tpm.layers['tpm'] = np.array(adata_tpm.layers['tpm'])
    adata_tpm.write_h5ad(f'{basepath}adata_{simname}_tpm.h5ad')

In [8]:
save_tpm_mtx_and_tpm_adata(adata_sim,simname='sim')
save_tpm_mtx_and_tpm_adata(adata_sim_only_marker_genes_DE,simname='sim_only_marker_genes_DE')

sim: subsetting adata to genes with known length, new shape: (23822, 28151)


tcmalloc: large alloc 2682454016 bytes == 0x362106000 @ 
tcmalloc: large alloc 5364908032 bytes == 0x401f36000 @ 
tcmalloc: large alloc 5364908032 bytes == 0x7e0606000 @ 
tcmalloc: large alloc 5364908032 bytes == 0x9600fe000 @ 
tcmalloc: large alloc 5364908032 bytes == 0x7e0606000 @ 


sim_only_marker_genes_DE: subsetting adata to genes with known length, new shape: (23822, 28151)


tcmalloc: large alloc 5364908032 bytes == 0x9600fe000 @ 
tcmalloc: large alloc 5364908032 bytes == 0x401f36000 @ 
